# 01 - EDA and Cleaning
Initial data exploration and cleaning steps.

In [12]:
import pandas as pd
import numpy as np
from scipy.sparse import coo_matrix, csr_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, Tuple, Optional
import ast
import gc # Garbage Collector để dọn rác RAM chủ động
import pickle
from pathlib import Path
from scipy import sparse
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.exceptions import NotFittedError


# CollaborativeFeatureEngineer Class

In [13]:
class CollaborativeFeatureEngineer:
    """
    Module xử lý Feature Engineering cho hệ thống gợi ý sử dụng ma trận thưa.
    Tối ưu hóa cho dữ liệu lớn và bộ nhớ RAM hạn chế.
    """

    def __init__(self):
        self.user_to_idx: Dict = {}
        self.idx_to_user: Dict = {}
        self.movie_to_idx: Dict = {}
        self.idx_to_movie: Dict = {}
        self.user_means: Optional[np.ndarray] = None
        self.item_means: Optional[np.ndarray] = None
        self.matrix_normalized: Optional[csr_matrix] = None

    def fit_transform(self, df: pd.DataFrame) -> csr_matrix:
        """
        Thực hiện toàn bộ quy trình từ mapping đến chuẩn hóa.
        
        Args:
            df: DataFrame chứa columns ['userId', 'movieId', 'rating']
            
        Returns:
            csr_matrix: Ma trận đã được chuẩn hóa (Double-centered).
        """
        # Bước 1: Mapping IDs
        row_indices, col_indices = self._create_mappings(df)    
        
        # Bước 2: Build Sparse Matrix
        # Sử dụng float32 để tiết kiệm 50% RAM so với float64
        ratings = df['rating'].values.astype(np.float32)
        sparse_mat = coo_matrix(
            (ratings, (row_indices, col_indices)),
            shape=(len(self.user_to_idx), len(self.movie_to_idx))
        ).tocsr()

        # Bước 3: Double-Centering
        self.matrix_normalized = self._apply_double_centering(sparse_mat)
        
        return self.matrix_normalized

    def _create_mappings(self, df: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
        """Tạo từ điển ánh xạ hai chiều và trả về mảng index."""
        unique_users = df['userId'].unique()
        unique_movies = df['movieId'].unique()

        self.user_to_idx = {uid: i for i, uid in enumerate(unique_users)}
        self.idx_to_user = {i: uid for uid, i in self.user_to_idx.items()}
        
        self.movie_to_idx = {mid: i for i, mid in enumerate(unique_movies)}
        self.idx_to_movie = {i: mid for mid, i in self.movie_to_idx.items()}

        row_indices = df['userId'].map(self.user_to_idx).values.astype(np.int32)
        col_indices = df['movieId'].map(self.movie_to_idx).values.astype(np.int32)
        
        return row_indices, col_indices

    def _apply_double_centering(self, matrix: csr_matrix) -> csr_matrix:
        """Khử bias User và Item trên cấu trúc thưa."""
        # Chế độ tính toán tránh biến đổi ma trận thưa thành ma trận dày (dense)
        mat = matrix.copy()
        
        # 1. Row-wise mean centering (User bias)
        # Chỉ tính trung bình trên các phần tử khác 0
        row_sums = mat.sum(axis=1).A1
        row_counts = np.diff(mat.indptr)
        self.user_means = np.divide(row_sums, row_counts,out=np.zeros_like(row_sums, dtype=np.float32), where=row_counts > 0)
        
        # Trừ trung bình hàng
        mat.data -= np.repeat(self.user_means, row_counts)

        # 2. Column-wise mean centering (Item bias)
        mat_csc = mat.tocsc()
        col_sums = mat_csc.sum(axis=0).A1
        col_counts = np.diff(mat_csc.indptr)
        self.item_means = np.divide(col_sums, col_counts,out=np.zeros_like(col_sums, dtype=np.float32), where=col_counts > 0)
        
        # Trừ trung bình cột
        mat_csc.data -= np.repeat(self.item_means, col_counts)
        
        return mat_csc.tocsr()

    def get_sparsity(self) -> float:
        """Tính toán phần trăm độ thưa của ma trận."""
        if self.matrix_normalized is None:
            return 0.0
        n_elements = self.matrix_normalized.shape[0] * self.matrix_normalized.shape[1]
        n_nonzero = self.matrix_normalized.nnz
        sparsity = (1 - n_nonzero / n_elements) * 100
        return sparsity

    def visualize_distribution(self, original_ratings: pd.Series):
        """Vẽ biểu đồ phân phối trước và sau chuẩn hóa."""
        plt.figure(figsize=(12, 5))
        
        plt.subplot(1, 2, 1)
        sns.histplot(original_ratings, bins=10, kde=True, color='blue')
        plt.title("Original Rating Distribution")
        
        plt.subplot(1, 2, 2)
        sns.histplot(self.matrix_normalized.data, bins=50, kde=True, color='red')
        plt.title("Double-Centered Rating Distribution")
        plt.show()

    def plot_top_interactions_heatmap(self, k: int = 50):
        """Vẽ Heatmap cho Top-K User và Item có nhiều tương tác nhất."""
        # Tìm top users/items dựa trên số lượng tương tác (nnz)
        row_counts = np.diff(self.matrix_normalized.indptr)
        top_users = np.argsort(row_counts)[-k:]
        
        col_counts = np.diff(self.matrix_normalized.tocsc().indptr)
        top_items = np.argsort(col_counts)[-k:]
        
        # Trích xuất ma trận con (Sub-matrix)
        sub_mat = self.matrix_normalized[top_users, :][:, top_items].toarray()
        
        plt.figure(figsize=(10, 8))
        sns.heatmap(sub_mat, cmap='RdBu_r', center=0)
        plt.title(f"Heatmap of Top {k}x{k} Interactions (Normalized)")
        plt.xlabel("Movie Index")
        plt.ylabel("User Index")
        plt.show()

    def get_meta_data(self) -> Dict:
        """Xuất dữ liệu mapping và các giá trị bias."""
        return {
            "user_mapping": self.user_to_idx,
            "movie_mapping": self.movie_to_idx,
            "user_means": self.user_means,
            "item_means": self.item_means
        }

# ContentBasedRecommender Class

In [14]:
class ContentBasedRecommender:
    """
    Hệ thống gợi ý dựa trên nội dung sử dụng TF-IDF Vectorization.
    Tối ưu hóa cho các tập dữ liệu metadata phim cỡ trung/lớn.
    """
    
    def __init__(self, max_features=10000, min_df=5, max_df=0.8):
        """
        Khởi tạo Recommender Pipeline.
        
        Args:
            max_features (int): Số lượng từ vựng tối đa.
            min_df (int/float): Cắt bỏ các từ xuất hiện dưới ngưỡng này.
            max_df (float): Cắt bỏ các từ xuất hiện trên tỷ lệ này (corpus-specific stopwords).
        """
        self._download_nltk_resources()
        
        self.stop_words = set(stopwords.words('english'))
        self.lemmatizer = WordNetLemmatizer()
        
        # Khởi tạo TF-IDF Vectorizer với cấu hình tối ưu bộ nhớ
        self.vectorizer = TfidfVectorizer(
            max_features=max_features,
            min_df=min_df,
            max_df=max_df,
            stop_words='english', # Lớp bảo vệ thứ 2 từ sklearn
            dtype=np.float32      # Giảm một nửa RAM so với float64 mặc định
        )
        self.tfidf_matrix = None
        
    def _download_nltk_resources(self):
        """Tải các tài nguyên NLTK cần thiết một cách an toàn."""
        resources = ['stopwords', 'wordnet', 'omw-1.4']
        for res in resources:
            try:
                nltk.data.find(f'corpora/{res}')
            except LookupError:
                nltk.download(res, quiet=True)

    def _preprocess_text_series(self, text_series: pd.Series) -> pd.Series:
        """
        Tiền xử lý dữ liệu văn bản dạng vectorized.
        
        Quy trình: Xử lý NaN -> Lowercase -> Xóa ký tự đặc biệt -> Xóa stopwords & Lemmatize.
        """
        # 1. Xử lý null và ép kiểu an toàn (vectorized)
        cleaned_series = text_series.fillna("").astype(str)
        
        # 2. Lowercase (vectorized)
        cleaned_series = cleaned_series.str.lower()
        
        # 3. Xóa ký tự không phải chữ/số (vectorized bằng regex)
        cleaned_series = cleaned_series.str.replace(r'[^a-z0-9\s]', ' ', regex=True)
        
        # 4. Tokenize, Stopwords removal và Lemmatization bằng apply (nhanh nhất có thể trong pandas)
        # Pre-bind method lookup để tối ưu hóa tốc độ bên trong lambda
        lemmatize = self.lemmatizer.lemmatize
        stopwords_set = self.stop_words
        
        def process_tokens(text):
            return " ".join([
                lemmatize(word) for word in text.split() 
                if word not in stopwords_set and len(word) > 1
            ])
            
        cleaned_series = cleaned_series.apply(process_tokens)
        
        return cleaned_series

    def fit_transform(self, df: pd.DataFrame, feature_col: str):
        """
        Huấn luyện vectorizer và biến đổi text corpus thành TF-IDF matrix.
        
        Args:
            df (pd.DataFrame): DataFrame chứa dữ liệu phim.
            feature_col (str): Tên cột chứa text gộp.
            
        Returns:
            scipy.sparse.csr_matrix: Ma trận thưa TF-IDF.
        """
        if feature_col not in df.columns:
            raise ValueError(f"Cột '{feature_col}' không tồn tại trong DataFrame.")
            
        print("Bắt đầu tiền xử lý văn bản...")
        processed_text = self._preprocess_text_series(df[feature_col])
        
        print("Bắt đầu xây dựng ma trận TF-IDF...")
        self.tfidf_matrix = self.vectorizer.fit_transform(processed_text)
        
        vocab_size = len(self.vectorizer.vocabulary_)
        print(f"Hoàn thành. Kích thước từ vựng thực tế: {vocab_size}")
        print(f"Kích thước ma trận TF-IDF: {self.tfidf_matrix.shape}")
        
        return self.tfidf_matrix

    def get_feature_names(self):
        """Lấy danh sách các từ vựng đã được học."""
        try:
            return self.vectorizer.get_feature_names_out()
        except AttributeError:
            raise NotFittedError("Vectorizer chưa được huấn luyện. Hãy gọi fit_transform trước.")

# Hướng dẫn tích hợp:
# recommender = ContentBasedRecommender(max_features=15000, min_df=5, max_df=0.8)
# tfidf_matrix = recommender.fit_transform(df, 'content_feature')

# ETL

In [ ]:


class MovieDataPipeline:
    def __init__(self, metadata_path, ratings_path, keywords_path=None, credits_path=None):
        """
        Khởi tạo Pipeline với đường dẫn file.
        """
        self.metadata_path = metadata_path
        self.ratings_path = ratings_path
        self.keywords_path = keywords_path
        self.credits_path = credits_path
        self.movies_df = None
        self.ratings_df = None
        self.collab_matrix = None

    @staticmethod
    def _safe_literal_eval(value):
        if pd.isna(value):
            return []
        try:
            parsed = ast.literal_eval(value)
            return parsed if isinstance(parsed, list) else []
        except Exception:
            return []

    @staticmethod
    def _extract_names(json_like_str):
        items = MovieDataPipeline._safe_literal_eval(json_like_str)
        return ' '.join(str(item.get('name', '')).strip() for item in items if item.get('name'))

    @staticmethod
    def _extract_credits_text(cast_str, crew_str):
        cast_items = MovieDataPipeline._safe_literal_eval(cast_str)
        crew_items = MovieDataPipeline._safe_literal_eval(crew_str)

        top_cast = [
            str(member.get('name', '')).strip()
            for member in cast_items[:5]
            if member.get('name')
        ]
        directors = [
            str(member.get('name', '')).strip()
            for member in crew_items
            if str(member.get('job', '')).lower() == 'director' and member.get('name')
        ]
        return ' '.join(top_cast + directors)

    def load_and_optimize(self):
        """
        Nạp dữ liệu kèm kỹ thuật Read-time Downcasting (Ép kiểu ngay khi đọc).
        Đây là kỹ thuật sống còn để chạy dữ liệu lớn trên laptop.
        """
        print("1. Đang nạp và tối ưu hóa bộ nhớ...")
        
        # Chỉ nạp những cột thật sự cần thiết. Bỏ qua các cột rác như homepage, tagline...
        metadata_cols = ['id', 'title', 'genres']
        
        # Đọc metadata, xử lý lỗi dòng (error_bad_lines) do file csv gốc thỉnh thoảng lỗi format
        self.movies_df = pd.read_csv(self.metadata_path, usecols=metadata_cols, low_memory=False)
        
        # Xử lý cột ID của movies: Loại bỏ các ID không phải là số (dữ liệu nhiễu) và ép về int32
        self.movies_df['id'] = pd.to_numeric(self.movies_df['id'], errors='coerce')
        self.movies_df = self.movies_df.dropna(subset=['id'])
        self.movies_df['id'] = self.movies_df['id'].astype('int32')

        # Nạp và tạo cột keywords từ keywords.csv
        if self.keywords_path is not None and Path(self.keywords_path).exists():
            keywords_df = pd.read_csv(self.keywords_path, usecols=['id', 'keywords'], low_memory=False)
            keywords_df['id'] = pd.to_numeric(keywords_df['id'], errors='coerce')
            keywords_df = keywords_df.dropna(subset=['id'])
            keywords_df['id'] = keywords_df['id'].astype('int32')
            keywords_df['keywords'] = keywords_df['keywords'].apply(self._extract_names)
            self.movies_df = self.movies_df.merge(keywords_df[['id', 'keywords']], on='id', how='left')
        else:
            self.movies_df['keywords'] = ''

        # Nạp và tạo cột credits từ credits.csv (gồm top cast + director)
        if self.credits_path is not None and Path(self.credits_path).exists():
            credits_df = pd.read_csv(self.credits_path, usecols=['id', 'cast', 'crew'], low_memory=False)
            credits_df['id'] = pd.to_numeric(credits_df['id'], errors='coerce')
            credits_df = credits_df.dropna(subset=['id'])
            credits_df['id'] = credits_df['id'].astype('int32')
            credits_df['credits'] = credits_df.apply(
                lambda row: self._extract_credits_text(row['cast'], row['crew']), axis=1
            )
            self.movies_df = self.movies_df.merge(credits_df[['id', 'credits']], on='id', how='left')
        else:
            self.movies_df['credits'] = ''

        self.movies_df['keywords'] = self.movies_df['keywords'].fillna('')
        self.movies_df['credits'] = self.movies_df['credits'].fillna('')

        # Đọc Ratings: Chỉ định trước Data Types (Kỹ thuật ăn tiền với dân C++)
        # Thay vì tốn 64 bits cho rating, ta dùng float32. user_id và movie_id dùng int32
        rating_dtypes = {
            'userId': 'int32',
            'movieId': 'int32',
            'rating': 'float32'
        }
        # Chỉ lấy 3 cột, bỏ qua 'timestamp' vì RecSys cơ bản chưa dùng tới Context-aware
        self.ratings_df = pd.read_csv(self.ratings_path, usecols=['userId', 'movieId', 'rating'], dtype=rating_dtypes)
        
        print(f"-> Đã nạp thành công! Ratings shape: {self.ratings_df.shape}")

    def clean_metadata(self):
        """
        Xử lý Missing Values và làm sạch Text cho Content-Based.
        """
        print("2. Đang làm sạch Metadata...")
        
        # Đảm bảo title luôn hợp lệ cho downstream hiển thị/recommendation.
        self.movies_df = self.movies_df.dropna(subset=['title'])
        
        # Cột genres trong dataset này đang ở dạng String của List of Dictionaries (VD: "[{'id': 12, 'name': 'Animation'}]")
        # Ta cần parse string này thành Python object bằng ast.literal_eval, sau đó rút trích tên thể loại.
        def extract_genres(x):
            try:
                # Trích xuất 'name' từ list các dict
                genres = ast.literal_eval(x)
                return ' '.join([i['name'] for i in genres])
            except:
                return ''
                
        self.movies_df['genres_text'] = self.movies_df['genres'].apply(extract_genres)
        self.movies_df = self.movies_df.drop(columns=['genres']) # Xóa cột cũ giải phóng RAM

    def prune_interactions(self, min_movie_ratings=10, min_user_ratings=5):
        """
        Cắt tỉa ma trận User-Item. Loại bỏ Users vãng lai và Phim quá chìm.
        """
        print(f"3. Đang cắt tỉa ma trận (Lọc User >= {min_user_ratings} rates, Movie >= {min_movie_ratings} rates)...")
        
        # Bước 1: Giữ các phim có từ min_movie_ratings đánh giá trở lên
        movie_counts = self.ratings_df['movieId'].value_counts()
        valid_movies = movie_counts[movie_counts >= min_movie_ratings].index
        self.ratings_df = self.ratings_df[self.ratings_df['movieId'].isin(valid_movies)]
        
        # Bước 2: Giữ các user có từ min_user_ratings đánh giá trở lên
        user_counts = self.ratings_df['userId'].value_counts()
        valid_users = user_counts[user_counts >= min_user_ratings].index
        self.ratings_df = self.ratings_df[self.ratings_df['userId'].isin(valid_users)]
        
        # Chỉ định lại index để dọn dẹp RAM dư thừa từ Pandas View
        self.ratings_df = self.ratings_df.reset_index(drop=True)
        gc.collect() # Ép Garbage Collector chạy tay để dọn dẹp biến tạm
        
        print(f"-> Sau cắt tỉa: Ratings shape thu gọn còn {self.ratings_df.shape}")

    def extract_features(self):
        """
        Chuẩn bị Feature cuối cùng cho 2 nhánh mô hình.
        """
        print("4. Chuẩn bị đặc trưng (Feature Extraction)...")
        
        # Đảm bảo dữ liệu Ratings chỉ tham chiếu tới các Phim có tồn tại trong Metadata
        valid_metadata_ids = self.movies_df['id'].unique()
        self.ratings_df = self.ratings_df[self.ratings_df['movieId'].isin(valid_metadata_ids)]
        
        # Content-Based Feature: Gộp Text
        # Content-based models (như TF-IDF hoặc Word2Vec) cần 1 chuỗi văn bản tổng hợp.
        self.movies_df['content_feature'] = (
            self.movies_df['title'] + " " + self.movies_df['genres_text'] + " " + self.movies_df['keywords'] + " " + self.movies_df['credits']
        )
        
    def show_statistics(self):
        """
        In ra Thống kê mô tả (Descriptive Statistics).
        """
        print("\n=== THỐNG KÊ MÔ TẢ TỔNG QUAN ===")
        print("1. Phân phối điểm đánh giá (Ratings Distribution):")
        print(self.ratings_df['rating'].describe())
        
        sparsity = 1.0 - (len(self.ratings_df) / (self.ratings_df['userId'].nunique() * self.ratings_df['movieId'].nunique()))
        print(f"\n2. Độ thưa của ma trận (Sparsity): {sparsity * 100:.4f}%")
        print("   (Điều này có nghĩa là {:.4f}% ô trong ma trận đang bị trống - Rất bình thường trong RecSys!)".format(sparsity * 100))

    def build_collaborative_artifacts(self, output_dir):
        """
        Dùng CollaborativeFeatureEngineer để tạo ma trận thưa chuẩn hóa
        và lưu artifacts phục vụ huấn luyện/inference collaborative.
        """
        print("5. Đang tối ưu dữ liệu Collaborative bằng CollaborativeFeatureEngineer...")

        engineer = CollaborativeFeatureEngineer()
        self.collab_matrix = engineer.fit_transform(self.ratings_df)

        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)

        collab_matrix_path = output_dir / 'collab_matrix_normalized.npz'
        collab_meta_path = output_dir / 'collab_mappings.pkl'

        sparse.save_npz(collab_matrix_path, self.collab_matrix)
        with open(collab_meta_path, 'wb') as f:
            pickle.dump(engineer.get_meta_data(), f)

        print(f"-> Collaborative matrix shape: {self.collab_matrix.shape}, nnz={self.collab_matrix.nnz}")
        print(f"-> Sparsity sau chuẩn hóa: {engineer.get_sparsity():.4f}%")
        print(f"-> Đã lưu collaborative artifacts:\n- Matrix: {collab_matrix_path}\n- Metadata: {collab_meta_path}")

    def build_collaborative_artifacts(self, output_dir):
        """
        Dùng CollaborativeFeatureEngineer để tạo ma trận thưa chuẩn hóa
        và lưu artifacts phục vụ huấn luyện/inference collaborative.
        """
        print("6. Đang tối ưu dữ liệu Collaborative bằng CollaborativeFeatureEngineer...")

        engineer = CollaborativeFeatureEngineer()
        self.collab_matrix = engineer.fit_transform(self.ratings_df)

        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)

        collab_matrix_path = output_dir / 'collab_matrix_normalized.npz'
        collab_meta_path = output_dir / 'collab_mappings.pkl'

        sparse.save_npz(collab_matrix_path, self.collab_matrix)
        with open(collab_meta_path, 'wb') as f:
            pickle.dump(engineer.get_meta_data(), f)

        print(f"-> Collaborative matrix shape: {self.collab_matrix.shape}, nnz={self.collab_matrix.nnz}")
        print(f"-> Sparsity sau chuẩn hóa: {engineer.get_sparsity():.4f}%")
        print(f"-> Đã lưu collaborative artifacts:\n- Matrix: {collab_matrix_path}\n- Metadata: {collab_meta_path}")

    def build_tfidf_artifacts(self, output_dir, source_df=None, feature_col='content_feature', max_features=10000, min_df=1, max_df=0.8):
        """
        Huấn luyện ContentBasedRecommender và lưu TF-IDF artifacts (.npz, .pkl).

        Args:
            output_dir: Thư mục đầu ra chứa artifacts.
            source_df: DataFrame đầu vào. Nếu None thì dùng self.movies_df.
            feature_col: Tên cột text để vector hóa.
            max_features, min_df, max_df: Cấu hình cho ContentBasedRecommender.

        Returns:
            tfidf_matrix: Ma trận TF-IDF dạng sparse.
            recommender: Instance ContentBasedRecommender đã fit.
        """
        print("7. Đang xây dựng TF-IDF artifacts bằng ContentBasedRecommender...")
        data_df = self.movies_df if source_df is None else source_df

        if data_df is None:
            raise ValueError('Khong co du lieu dau vao. Hay truyen source_df hoac goi extract_features truoc.')

        if feature_col not in data_df.columns:
            raise ValueError(f"Cột '{feature_col}' không tồn tại trong DataFrame đầu vào.")

        print("Dang xay dung TF-IDF artifacts...")
        recommender = ContentBasedRecommender(max_features=max_features, min_df=min_df, max_df=max_df)
        tfidf_matrix = recommender.fit_transform(data_df, feature_col=feature_col)

        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)

        save_npz_path = output_dir / 'tfidf_matrix.npz'
        save_pkl_path = output_dir / 'tfidf_matrix.pkl'

        sparse.save_npz(save_npz_path, tfidf_matrix)
        with open(save_pkl_path, 'wb') as f:
            pickle.dump(tfidf_matrix, f)

        print(f"Da luu tfidf_matrix vao: {save_npz_path}")
        print(f"Da luu tfidf_matrix vao: {save_pkl_path}")

        return tfidf_matrix, recommender

# --- CÁCH SỬ DỤNG PIPELINE ---
if __name__ == "__main__":
    # Tự động suy ra root project để chạy được từ mọi thư mục hiện hành
    project_root = next(
        (path for path in [Path.cwd(), *Path.cwd().parents]
         if (path / "data" / "processed" / "clean_movies.parquet").exists()),
        None,
)

    if project_root is None:
        raise FileNotFoundError("Không tìm thấy data/processed/clean_movies.parquet trong thư mục hiện tại hoặc các thư mục cha.")

    METADATA_PATH = project_root / 'data' / 'raw' / 'movies_metadata.csv'
    RATINGS_PATH = project_root / 'data' / 'raw' / 'ratings.csv'  # Có thể thử với ratings_small.csv trước để test logic
    KEYWORDS_PATH = project_root / 'data' / 'raw' / 'keywords.csv'
    CREDITS_PATH = project_root / 'data' / 'raw' / 'credits.csv'
    processed_dir = project_root / 'data' / 'processed'
    processed_dir.mkdir(parents=True, exist_ok=True)
    
    # Khởi tạo và chạy tuần tự
    pipeline = MovieDataPipeline(METADATA_PATH, RATINGS_PATH, KEYWORDS_PATH, CREDITS_PATH)
    pipeline.load_and_optimize()
    pipeline.clean_metadata()
    pipeline.prune_interactions(min_movie_ratings=10, min_user_ratings=5)
    pipeline.extract_features()
    pipeline.show_statistics()
    # Mẹo nhỏ: Lưu lại dưới dạng parquet (nhanh hơn, lưu trữ cấu trúc type int32/float32 tốt hơn CSV)
    clean_ratings_path = processed_dir / 'clean_ratings.parquet'
    clean_movies_path = processed_dir / 'clean_movies.parquet'
    pipeline.ratings_df.to_parquet(clean_ratings_path)
    pipeline.movies_df.to_parquet(clean_movies_path)
    print(f"\n5. Đã lưu dữ liệu xử lý:")
    print(f"- Ratings: {clean_ratings_path}")
    print(f"- Movies: {clean_movies_path}")
    pipeline.build_collaborative_artifacts(processed_dir)
    pipeline.build_tfidf_artifacts(processed_dir)
    
    
    

1. Đang nạp và tối ưu hóa bộ nhớ...
-> Đã nạp thành công! Ratings shape: (26024289, 3)
2. Đang làm sạch Metadata...
3. Đang cắt tỉa ma trận (Lọc User >= 5 rates, Movie >= 10 rates)...
-> Sau cắt tỉa: Ratings shape thu gọn còn (25916033, 3)
4. Chuẩn bị đặc trưng (Feature Extraction)...

=== THỐNG KÊ MÔ TẢ TỔNG QUAN ===
1. Phân phối điểm đánh giá (Ratings Distribution):
count    1.141235e+07
mean     3.532923e+00
std      1.066535e+00
min      5.000000e-01
25%      3.000000e+00
50%      4.000000e+00
75%      4.000000e+00
max      5.000000e+00
Name: rating, dtype: float64

2. Độ thưa của ma trận (Sparsity): 99.1643%
   (Điều này có nghĩa là 99.1643% ô trong ma trận đang bị trống - Rất bình thường trong RecSys!)

5. Đã lưu dữ liệu xử lý:
- Ratings: e:\Year_3\Semester_2\Machine Learning\BTL\video_rec_system\data\processed\clean_ratings.parquet
- Movies: e:\Year_3\Semester_2\Machine Learning\BTL\video_rec_system\data\processed\clean_movies.parquet
6. Đang tối ưu dữ liệu Collaborative bằng Co

# Get Data

In [21]:
project_root = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "data" / "processed" / "clean_movies.parquet").exists()),
    None,
)

if project_root is None:
    raise FileNotFoundError("Không tìm thấy data/processed/clean_movies.parquet trong thư mục hiện tại hoặc các thư mục cha.")

clean_movies_file_path = project_root / "data" / "processed" / "clean_movies.parquet"
clean_ratings_file_path = project_root / "data" / "processed" / "clean_ratings.parquet"
clean_movies_df = pd.read_parquet(clean_movies_file_path)
clean_ratings_df = pd.read_parquet(clean_ratings_file_path)

print(clean_movies_df.head())
print(clean_ratings_df.head())

      id                        title  \
0    862                    Toy Story   
1   8844                      Jumanji   
2  15602             Grumpier Old Men   
3  31357            Waiting to Exhale   
4  11862  Father of the Bride Part II   

                                            keywords  \
0  jealousy toy boy friendship friends rivalry bo...   
1  board game disappearance based on children's b...   
2   fishing best friend duringcreditsstinger old men   
3  based on novel interracial relationship single...   
4  baby midlife crisis confidence aging daughter ...   

                                             credits  \
0  Tom Hanks Tim Allen Don Rickles Jim Varney Wal...   
1  Robin Williams Jonathan Hyde Kirsten Dunst Bra...   
2  Walter Matthau Jack Lemmon Ann-Margret Sophia ...   
3  Whitney Houston Angela Bassett Loretta Devine ...   
4  Steve Martin Diane Keaton Martin Short Kimberl...   

                genres_text                                    content_feature 

# Summarize of Data 

collab_matrix_normalized.npz được tối ưu từ clean_movies.parquet

tfidf_matrix.npz được tối ưu từ clean_ratings.parquet


Mô tả dữ liệu được xử lý:
movies_df: DataFrame chứa thông tin phim đã được làm sạch, với các cột như id, title, genres_text, keywords, credits, content_feature (chuỗi tổng hợp cho Content-Based)
ratings_df: DataFrame chứa thông tin đánh giá đã được cắt tỉa, với các cột userId, movieId, rating (đã ép kiểu int32/float32 để tiết kiệm RAM)
id là khóa chính để liên kết giữa movies_df và ratings_df. Các cột genres_text, keywords, credits đã được trích xuất và làm sạch để phục vụ cho mô hình Content-Based Filtering. Ma trận User-Item đã được cắt tỉa để loại bỏ các user vãng lai và phim quá chìm, giúp tăng hiệu quả của mô hình Collaborative Filtering.
genres_text: Chuỗi chứa tên các thể loại phim, được trích xuất từ cột genres gốc (dạng string của list of dicts).
keywords: Chuỗi chứa tên các từ khóa liên quan đến phim (Nội dung phim)
credits: Chuỗi chứa tên các diễn viên chính và đạo diễn (Thông tin về dàn diễn viên và đạo diễn)
content_feature: Chuỗi tổng hợp từ genres_text + keywords + credits, được sử dụng làm đặc trưng đầu vào cho mô hình Content-Based Filtering.
userId: ID của người dùng (đã ép kiểu int32)
movieId: ID của phim (đã ép kiểu int32)
rating: Điểm đánh giá của user cho movie (đã ép kiểu float32)
